# Guía L1 — La factura de la ciudad · **Sesión 2: cobrar el mes**

**Cálculo Integral — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

> La sesión 1 midió la energía; hoy la volvemos plata. Y la plata, van a ver, también se acumula — pero con mañas.

**Antes de empezar:** copia propia del cuaderno, la tarea puente lista (los 9 casos verificados), y **el recibo de energía de su casa** sobre la mesa.

---
## ⏸️ Momento 1 — Predecir

P4 y P5 están en la guía. **En papel, a mano.** La apuesta de P5 —¿cobrar el total del mes da lo mismo que cobrar día a día y sumar?— es la protagonista de hoy. Foto al informe antes de seguir.

---
## 1 · La tarifa en vivo: nuestra primera API

La sesión pasada el mesero nos trajo un plato congelado. Hoy pedimos en vivo: la URL de abajo es un **pedido** al portal de datos abiertos del Estado colombiano (`datos.gov.co`), y la respuesta es el pliego tarifario oficial de la CHEC — el mismo que decide cuánto paga su casa.

Detalle de la casa: las APIs sirven porciones. Sin el `$limit`, el mesero trae solo las primeras 1.000 filas y uno ni se entera. Leer la documentación del dataset **es** parte de leer el dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_TARIFAS = "https://www.datos.gov.co/resource/c698-wqig.csv?$limit=60000"
# Si el wifi del laboratorio se cae, el monitor tiene la copia congelada:
# RUTA_TARIFAS = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/chec_tarifas_congeladas.csv"

tarifas = pd.read_csv(RUTA_TARIFAS)
print("Filas y columnas:", tarifas.shape)
print(list(tarifas.columns))

In [ ]:
# ✏️ Ejercicio 1 — El ritual, ahora sin acompañamiento: mirar antes de cruzar.
# Este dataset tiene más columnas y más trampas que el de Londres. Respondan con código:
#   a) ¿Qué mercados (mercado_nombre) aparecen y con cuántas filas cada uno?
#   b) ¿Qué años cubre el pliego? (ojo: la columna de año puede venir como texto "2,026")
#   c) ¿Qué estratos residenciales hay y qué valores toma el "rango de consumo"?
#   d) ¿Hay NaN donde no debería haberlos?

# Normalizamos el año de una vez (viene con coma de miles en algunas descargas):
tarifas["ano"] = tarifas["costounitariotarifa_ano"].astype(str).str.replace(",", "").astype(int)
tarifas["valor"] = pd.to_numeric(tarifas["costounitariotarifa_valor_tarifa"], errors="coerce")

print(tarifas["mercado_nombre"].value_counts())
# ... su turno con b), c), d)

---
## 2 · La tarifa también es una escalonada — del tiempo

En la sesión 1, la escalonada vivía en el eje del tiempo con escalones de un día. Miren ahora el costo unitario de un estrato a lo largo de los años: escalones de un **mes**, medidos en \$/kWh. El mundo institucional está hecho de funciones constantes a trozos.

In [ ]:
# La serie del estrato 3, rango subsidiado, mercado CHEC, condiciones estándar:
filtro = ((tarifas["mercado_nombre"] == "CHEC") &
          (tarifas["costounitariotarifa_tipomercado"] == "R") &
          (tarifas["nivel_tension_codigo"] == 1) &
          (tarifas["modalidadcobro_propiedad"] == "0% OR") &
          (tarifas["claseservicio_estrato"] == 3.0) &
          (tarifas["costounitariotarifa_rangoconsumo"] == 1))
serie = (tarifas[filtro]
         .assign(mes_num=lambda x: pd.to_numeric(x["costounitariotarifa_mes"]))
         .groupby(["ano", "mes_num"])["valor"].median()
         .reset_index()
         .sort_values(["ano", "mes_num"]))
serie["orden"] = range(len(serie))

plt.figure(figsize=(11, 4))
plt.step(serie["orden"], serie["valor"], where="post")
etiquetas = serie.iloc[::24]
plt.xticks(etiquetas["orden"], etiquetas["ano"], rotation=0)
plt.xlabel("Tiempo (un escalón por mes)")
plt.ylabel("Tarifa estrato 3, rango subsidiado ($/kWh)")
plt.title("La tarifa de la CHEC: otra función escalonada, ahora en pesos")
plt.show()

# 🗣️ Escriban la partición y la regla de ESTA escalonada. ¿Qué duran sus escalones?
#    ¿Qué unidades tiene? ¿Qué evento económico creen que explica el salto más grande?

---
## 3 · La tarifa como función: `valor_factura`

Del pliego a la función definida por trozos que decide cuánto paga una casa. Dos ingredientes reales:

- El **consumo de subsistencia** (CS): en Manizales (2.150 m; la norma fija el umbral en 1.000 m de altitud) son **130 kWh/mes** (173 kWh/mes por debajo de los 1.000 m). Hasta ahí, los estratos 1–3 pagan tarifa subsidiada; de ahí en adelante, todos pagan tarifa plena.
- El **pliego del mes**: lo sacamos del dataset, no de un número inventado. (El pliego trae variantes por propiedad de la red y nivel de tensión; tomamos las condiciones más comunes y la *mediana* cuando queda más de un valor — decisión de modelación, y queda escrita.)

In [ ]:
ANO_TARIFA, MES_TARIFA = 2026, 8   # el pliego que usaremos para cobrar (el más reciente)
CS = 130                            # kWh/mes de subsistencia para alturas >= 1000 m (Manizales: 2.150 m); 173 por debajo de 1000 m

base = tarifas[(tarifas["mercado_nombre"] == "CHEC") &
               (tarifas["costounitariotarifa_tipomercado"] == "R") &
               (tarifas["nivel_tension_codigo"] == 1) &
               (tarifas["modalidadcobro_propiedad"] == "0% OR") &
               (tarifas["ano"] == ANO_TARIFA) &
               (pd.to_numeric(tarifas["costounitariotarifa_mes"]) == MES_TARIFA) &
               (tarifas["claseservicio_descri"].str.contains("Residencial Estrato", na=False))]

pliego = base.pivot_table(index="claseservicio_estrato",
                          columns="costounitariotarifa_rangoconsumo",
                          values="valor", aggfunc="median")
pliego.columns = ["subsidiada", "plena"][:len(pliego.columns)]
print("Pliego CHEC", ANO_TARIFA, "-", MES_TARIFA, "($/kWh):")
print(pliego.round(2))

def valor_factura(kwh, estrato):
    """Valor de la energía de un mes (sin cargo fijo, sin alumbrado): función por trozos."""
    t_sub  = pliego.loc[estrato, "subsidiada"]
    t_plena = pliego.loc[estrato, "plena"] if not pd.isna(pliego.loc[estrato].get("plena", np.nan)) else t_sub
    if estrato <= 3:
        basico = min(kwh, CS) * t_sub      # el tramo subsidiado, hasta el CS
        resto  = max(kwh - CS, 0) * t_plena
        return basico + resto
    else:
        return kwh * t_sub                 # estratos 4-6: un solo precio (4: pleno; 5-6: con contribución)

for kwh in [100, 130, 200, 300]:
    print(f"Estrato 3, {kwh} kWh → ${valor_factura(kwh, 3):,.0f}")

In [ ]:
# ✏️ Ejercicio 2 — La auditoría del recibo.
# Con el recibo de un integrante: consumo del mes (kWh), estrato, y valor pagado por ENERGÍA.
kwh_recibo   = 0      # ← su recibo
estrato_casa = 0      # ← su estrato
pago_real    = 0      # ← lo que dice el recibo

# print(f"El modelo dice: ${valor_factura(kwh_recibo, estrato_casa):,.0f}  |  El recibo dice: ${pago_real:,.0f}")

# 🗣️ ¿El modelo reproduce, aproximadamente, lo que su casa pagó? ¿Qué explica la diferencia?
#    (cargo fijo, alumbrado público, contribución, el mes del pliego...). Acaban de auditar
#    su propio modelo contra la realidad — anoten el veredicto.

---
## 4 · La decisión de modelación: ¿qué estrato le toca a Londres?

Para cobrarle a un hogar londinense necesitamos asignarle un estrato — y eso es **supuesto nuestro**, no dato. Abajo hay un criterio de ejemplo (cuantiles de consumo). Su equipo puede adoptarlo o cambiarlo, pero sea cual sea el criterio: **se defiende por escrito en tres líneas**. No hay criterio “correcto”; hay criterios defendibles y criterios que no se pensaron.

In [ ]:
# Cargamos Londres (la copia congelada de la sesión 1) y armamos la tabla mensual del mes objetivo.
RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/london_energy.csv.gz"   # la misma copia congelada de la sesión 1
df = pd.read_csv(RUTA)
df.columns = ["hogar", "fecha", "kwh"]
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.to_period("M")

MES_COBRO = pd.Period("2013-01")
diario = df[df["mes"] == MES_COBRO]
mensual = diario.groupby("hogar").agg(energia=("kwh", "sum"), dias=("kwh", "size"))
mensual = mensual[mensual["dias"] >= 28]   # solo meses con calendario razonable (¡decisión escrita!)
print("Hogares a facturar en", MES_COBRO, ":", len(mensual))

# ✏️ Criterio de ejemplo (cámbienlo si quieren): estratos por cuantiles de consumo del mes.
cortes = mensual["energia"].quantile([0, .25, .50, .70, .85, .95, 1.0]).values
mensual["estrato"] = pd.cut(mensual["energia"], bins=cortes, labels=[1, 2, 3, 4, 5, 6],
                            include_lowest=True).astype(int)
print(mensual["estrato"].value_counts().sort_index())

# 🗣️ Defensa del criterio (tres líneas, aquí): ¿por qué ESTE reparto y no otro?

---
## 5 · El plato fuerte: el orden de la acumulación importa

Dos maneras de cobrar el mismo mes, con la misma tarifa:

- **Método A (primero acumular, después cobrar):** $\;T\!\left(\int_{mes} s\right)$ — la energía total del mes pasa una vez por la tarifa. Así factura la CHEC.
- **Método B (cobrar cada día, después acumular):** construir la escalonada del **gasto diario** $g(t) = T\big(s(t) \cdot 30\big)/30$ — cada día se cobra como si ese ritmo durara el mes — y acumular $\int_{mes} g(t)\,dt$.

P5 preguntaba si dan lo mismo. Veámoslo con nuestro hogar de siempre — y luego con la ciudad entera.

In [ ]:
HOGAR = "MAC002515"
ESTRATO = int(mensual.loc[HOGAR, "estrato"])
s_dias = diario[diario["hogar"] == HOGAR].sort_values("fecha")["kwh"].to_numpy()
print(f"Hogar {HOGAR} | estrato asignado: {ESTRATO} | energía del mes: {s_dias.sum():.2f} kWh")

# Método A: primero la integral, después la tarifa.
metodo_A = valor_factura(s_dias.sum(), ESTRATO)

# Método B: la tarifa día a día, después la integral.
g = np.array([valor_factura(sk * 30, ESTRATO) / 30 for sk in s_dias])   # $/día
metodo_B = g.sum()   # ∫ g(t) dt con Δt = 1 día

print(f"Método A — T(∫s):  ${metodo_A:,.0f}")
print(f"Método B — ∫T(s):  ${metodo_B:,.0f}")
print(f"Diferencia:        ${metodo_A - metodo_B:,.0f}")

plt.figure(figsize=(11, 4))
plt.step(range(1, len(g) + 1), g, where="post")
plt.xlabel("Día de enero de 2013"); plt.ylabel("Gasto del día ($/día)")
plt.title("g(t): la escalonada del gasto — consumo por tarifa, día a día")
plt.show()

# 🗣️ ¿Por qué NO dan lo mismo — y por qué B salió MÁS CARO? Miren los dos extremos del mes:
#    cobrado día a día, cada día de ráfaga (¡36 kWh/día!) se cobra como si TODO el mes corriera
#    a ese ritmo: desborda el CS y paga tarifa plena. Cobrado el total, las ráfagas se promedian
#    con los días del viaje (~1.9) ANTES de tocar la tarifa, y buena parte vuelve al tramo
#    subsidiado. Una tarifa de tramos crecientes castiga las rachas: los TRAMOS rompen el
#    intercambio entre acumular y cobrar.

In [ ]:
# ✏️ Ejercicio 3 — La ciudad entera: ¿a quién le cambia más la factura según el orden?
def cobrar_dos_veces(grupo):
    est = int(mensual.loc[grupo.name, "estrato"])
    sk = grupo.sort_values("fecha")["kwh"].to_numpy()
    A = valor_factura(sk.sum(), est)
    B = sum(valor_factura(x * 30, est) / 30 for x in sk)
    return pd.Series({"A": A, "B": B, "dif": A - B, "estrato": est})

facturas = (diario[diario["hogar"].isin(mensual.index)]
            .groupby("hogar").apply(cobrar_dos_veces, include_groups=False))

plt.figure(figsize=(9, 4))
plt.hist(facturas["dif"], bins=60)
plt.xlabel("Método A − Método B ($)"); plt.ylabel("Hogares")
plt.title("El precio del orden de la acumulación, hogar por hogar")
plt.show()

extremo = facturas["dif"].abs().idxmax()
print("El hogar donde el orden más pesa:", extremo)
print(facturas.loc[extremo].round(0))

# 🗣️ Grafiquen la escalera de ese hogar extremo (código de la sesión 1) y expliquen con
#    sus tramos por qué el orden le pesa tanto. ¿Qué clase de hogar es: parejo o de rachas?

In [ ]:
# La apuesta de la sesión 1 se resuelve: ¿qué propiedad rompió la plata?
dobles_kwh   = 2 * s_dias.sum()
factura_x1   = valor_factura(s_dias.sum(), ESTRATO)
factura_x2   = valor_factura(dobles_kwh, ESTRATO)
print(f"Energía: {s_dias.sum():.1f} → {dobles_kwh:.1f} kWh (razón exacta: {dobles_kwh / s_dias.sum():.2f})")
print(f"Factura: ${factura_x1:,.0f} → ${factura_x2:,.0f} (razón: {factura_x2 / factura_x1:.3f})")

# 🗣️ La energía se duplicó limpiamente (∫2s = 2∫s: homogeneidad). La factura NO.
#    Conclusión para el informe, con símbolos: si la tarifa T no es lineal,
#        T(∫s) ≠ ∫T(s)   y   T(2·kwh) ≠ 2·T(kwh).
#    La integral es lineal; el mundo que la cobra, no siempre.

---
## 6 · La curva del contador

La factura solo publica el último punto de una historia que el contador escribe todos los días: la **acumulación** $A(d) = \sum_{k \le d} s_k \, \Delta t_k$.

In [ ]:
A = np.concatenate([[0], np.cumsum(s_dias)])   # A(0)=0, A(1), ..., A(31)

plt.figure(figsize=(11, 4.5))
plt.plot(range(len(A)), A, linewidth=2)
plt.scatter([len(A) - 1], [A[-1]], zorder=3)
plt.annotate(f"la factura solo ve esto: {A[-1]:.1f} kWh",
             xy=(len(A) - 1, A[-1]), xytext=(14, A[-1] * 0.75),
             arrowprops=dict(arrowstyle="->"))
plt.xlabel("Día de enero de 2013"); plt.ylabel("Energía acumulada A(d) (kWh)")
plt.title("La curva que el contador dibuja en silencio")
plt.show()

# 🗣️ Pregunta final del informe (formúlenla, no la desarrollen): en esta curva creciente,
#    ¿QUÉ información del consumo diario quedó escondida en la PENDIENTE?
#    Miren el tramo del viaje. El próximo laboratorio empieza exactamente ahí.

---
## 7 · El microdataset del curso

Cada estudiante sube **una fila** a la tabla compartida del curso: `(codigo_anonimo, mes, kwh, valor_pagado, estrato)` — su recibo, anonimizado.

🗣️ Discutan 10 minutos: ¿por qué la tabla tiene *esas* columnas y no otras? ¿Qué preguntas futuras permite responder esa estructura y cuáles cierra? Elegir la estructura de una tabla es decidir qué se va a poder preguntar — eso también es matemáticas.

In [ ]:
# La nube final: la ciudad facturada, con nosotros adentro.
colores = {1: "#8c510a", 2: "#d8b365", 3: "#f6e8c3", 4: "#c7eae5", 5: "#5ab4ac", 6: "#01665e"}
plt.figure(figsize=(10, 5))
for e in range(1, 7):
    sub = facturas[facturas["estrato"] == e].join(mensual["energia"])
    plt.scatter(sub["energia"], sub["A"], s=5, alpha=0.4, color=colores[e], label=f"Estrato {e}")

# ✏️ Los puntos del curso (llenen con las filas del microdataset — kWh y valor por energía):
curso_kwh   = []   # p. ej. [180, 95, 210, ...]
curso_valor = []   # p. ej. [148000, 52000, 175000, ...]
plt.scatter(curso_kwh, curso_valor, s=90, marker="*", color="crimson", zorder=3, label="Nosotros")

plt.xlabel("Energía del mes: ∫ s(t) dt  (kWh)")
plt.ylabel("Valor facturado — método A ($)")
plt.title("Londres facturado por la CHEC — y nosotros, entre miles de hogares")
plt.legend(markerscale=2)
plt.show()

# 🗣️ Ahí estamos, del otro lado del océano. Y desde hoy sabemos qué integral es cada punto.

---
## 🗣️ Momento 3 — Explicar

Una página del equipo, aquí mismo: las dos discrepancias más interesantes entre predicción y resultado; la explicación, con tramos y símbolos, de por qué el orden de la acumulación cambia la factura; la defensa del criterio de estratificación; y tres líneas sobre qué fue real y qué fue supuesto en toda la guía.

---
## 🤖 Protocolo de IA (recordatorio)

Las tres reglas del curso, siempre: intento propio antes de consultar; la IA no es la autoridad final; todo uso va a la bitácora individual. La **cacería puntuable** está en la guía: pídanle a una IA la energía total de un mes desde la tabla diaria, sin mostrarle esta guía, y audítenla — ¿qué hizo con los días faltantes? Un cálculo correcto sobre una partición incompleta es la mentira más elegante de la ciencia de datos.

**Entregables** (cierre de esta semana): cuaderno `L1_apellidos.ipynb` ejecutado de arriba a abajo, bitácora individual, fila del microdataset, y la tarea puente. La rúbrica está en la guía — el criterio que más pesa es el **producto**: qué significa $s_k \, \Delta t_k$, en la energía y en la plata.